# 1. Gemini API 첫 호출

**목표** — API 키로 Gemini를 호출하고, 돌아온 응답 객체 안에 무엇이 들어있는지 전부 확인한다.

**소요 시간** 약 50분

| 다루는 것 | |
| --- | --- |
| 0 | 환경 확인 |
| 1 | 첫 호출 · 쓸 수 있는 모델 찾기 · 재시도 헬퍼 |
| 2 | 응답 객체 뜯어보기 |
| 3 | `system_instruction` — 역할 부여 |
| 4 | 에러 체험 |
| 5 | 연습문제 |
| 6 | 같은 호출을 OpenAI SDK로 |

> 개념이 헷갈리면 `[배포용] 1_LLM API 동작 원리와 토큰·과금.md`를 먼저 읽는다.

## 0. 환경 확인

`.env`에서 키를 읽어온다. **키 값 자체는 절대 출력하지 않는다** — 노트북에는 실행 결과가 저장되므로, 한 번 찍히면 파일을 공유할 때 딸려나간다.

In [28]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

# 존재 여부와 길이만 확인 (값은 출력하지 않는다)
key = os.getenv("GEMINI_API_KEY")
print("GEMINI_API_KEY:", "OK" if key else "없음 — [배포용] 0_실습 환경 구성과 API 키 준비.md 4절 참고", f"(길이 {len(key) if key else 0})")

GEMINI_API_KEY: OK (길이 53)


## 1. 첫 호출

`google-genai` SDK로 클라이언트를 만들고 질문을 보낸다. 딱 세 줄이다.

- `genai.Client(api_key=...)` — 어느 계정으로 호출할지 정한다. 한 번만 만들어두고 계속 쓴다.
- `client.models.generate_content(...)` — 실제 호출. 여기서 네트워크 요청이 나간다.
- `response.text` — 생성된 텍스트

모델은 **`gemini-3.1-flash-lite`** 를 쓴다. 응답이 빠르고 무료 한도가 여유로운 편이라 실습 중 `429`가 덜 난다.

> **주의: 모델명은 자주 바뀐다.** 이 셀에서 `404`가 나면 교재가 낡은 것이지 여러분 잘못이 아니다. 바로 아래에서 쓸 수 있는 모델을 찾는 방법을 다룬다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODEL = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=MODEL,
    contents="대한민국의 수도는 어디야? 한 문장으로 답해줘.",
)

print(response.text)
```

In [29]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 응답 텍스트가 출력되면 성공

from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [30]:
MODEL = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=MODEL,
    contents="안녕?",
)

In [31]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='안녕하세요! 만나서 반가워요. 오늘 어떤 도움을 드릴까요?',
            thought_signature=b"\x12q\no\x01\x11M2\x0f&\x10\xd96p\xb6'>\xeb\xaf\xd3\xd8\x00\xfacr\xee\x10\xb1T\xff\x0e\xac\xf5*6\x18\xf15\xcc\xb2[s\x1fF'6\xa0y\x19\x814\xf1\x8cZ\xb93\xf7K:q%^v\xfa\xb7\xadd<\x03\xb0\xf7O\xa3\x9e\xb8+1\x94\xdfg@w\xa1\xe0\xc6\xe1B\x0eX\xe4\x8f\xf1\xc7\xf5l\xdd\x86...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.1-flash-lite',
  response_id='1r6Gaqo1-efV7w-Ix6_oBg',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=16,
    prompt_token_count=4,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        

In [32]:
response.text

'안녕하세요! 만나서 반가워요. 오늘 어떤 도움을 드릴까요?'

### 여기서 에러가 난다면

| 에러 | 원인 | 내 잘못인가 | 해결 |
| --- | --- | --- | --- |
| `API key not valid` | 키가 틀렸거나 `.env`가 안 읽힘 | 예 | 0번 셀이 `OK`였는지 확인, 키 재발급 |
| `429 RESOURCE_EXHAUSTED` | **내가** 무료 한도 초과 | 예 | 1분 기다렸다 재실행 (노트북 02에서 자세히) |
| `503 UNAVAILABLE` | **구글 서버가** 일시적 과부하 | 아니오 | 잠시 후 재시도. 내 할당량과 무관하다 |
| `404 NOT_FOUND` | 모델명이 없어졌거나 내 프로젝트에 미제공 | 아니오 | 바로 아래 "모델명이 안 먹힐 때" 참고 |
| `ModuleNotFoundError` | 커널이 `.venv`가 아님 | 예 | 우측 상단 커널 선택에서 `.venv` 지정 |

**`429`와 `503`은 성격이 완전히 다르다.** `429`는 내가 너무 많이 불렀다는 뜻이고, `503`은 서버가 붐빈다는 뜻이라 내 할당량은 그대로다. 둘 다 **일시적**이라 재시도로 해결되는데, 이게 실무에서 얼마나 흔한지는 바로 아래에서 다룬다.

### 모델명이 안 먹힐 때 — 실제로 쓸 수 있는 모델 확인하기

**Gemini 모델명은 자주 바뀐다.** 이 교재를 만드는 동안에도 원래 쓰려던 모델이 "신규 사용자에게는 더 이상 제공되지 않는다"며 `404`를 냈다.

더 헷갈리는 점: **`models.list()`에 보이는데도 호출하면 404가 나는 모델이 있다.** 목록에 있다는 건 "그런 모델이 존재한다"는 뜻이지, "내 프로젝트가 쓸 수 있다"는 뜻이 아니다.

그래서 **직접 호출해보는 게 유일하게 확실한 확인 방법**이다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# 후보 모델을 하나씩 실제로 호출해서, 내 키로 쓸 수 있는 것을 찾는다
candidates = [
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash",
    "gemini-3.6-flash",
    "gemini-2.0-flash",
]

for name in candidates:
    try:
        r = client.models.generate_content(model=name, contents="한 단어로: 대한민국 수도")
        u = r.usage_metadata
        print(f"OK    {name:24s} 답={r.text.strip()[:10]!r}  토큰 p/c/t={u.prompt_token_count}/{u.candidates_token_count}/{u.total_token_count}")
    except Exception as e:
        print(f"불가  {name:24s} {str(e)[:60]}")
```

In [33]:
candidates = [
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash",
    "gemini-3.6-flash",
    "gemini-2.0-flash",
]

In [36]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 모델마다 OK / 불가 가 갈리는지 본다
for name in candidates:
    r = client.models.generate_content(model=name, contents="한 단어로: 대한민국의 수도")
    u = response.usage_metadata
    print(f"OK    {name:24s} 답={r.text.strip()[:10]!r}  토큰 p/c/t={u.prompt_token_count}/{u.candidates_token_count}/{u.total_token_count}")

OK    gemini-3.1-flash-lite    답='서울'  토큰 p/c/t=4/16/20


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements.', 'status': 'NOT_FOUND'}}

In [ ]:
response.text

'안녕하세요! 만나서 반가워요. 오늘 어떤 도움을 드릴까요?'

In [ ]:
response.model_version

'gemini-3.1-flash-lite'

In [ ]:
response.usage_metadata.prompt_token_count

4

> **참고: `total`이 `prompt + candidates`보다 훨씬 큰 모델이 있다.** 답변 전에 내부적으로 "생각"하는 추론 모델이라, 그 사고 과정도 토큰으로 과금된다.
> 이 실습에서는 **응답이 빠르고 토큰 계산이 단순한 `gemini-3.1-flash-lite`** 를 쓴다. 위 셀에서 이게 `불가`로 나오면 `OK`인 다른 모델로 `MODEL`을 바꾼다.

### 일시적 오류에 대비하기 — 재시도 헬퍼

`429`와 `503`은 **실습 중에 실제로 자주 만난다.** 특히 아래처럼 반복 호출하는 셀은 한 번만 실패해도 셀 전체가 죽는다.

그래서 앞으로는 이 헬퍼를 통해 호출한다. **잠깐 기다렸다 다시 시도하되, 기다리는 시간을 점점 늘리는 방식(지수 백오프)** 이다. 같은 간격으로 재시도하면 회복되기 전에 또 때려서 상황이 더 나빠진다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
import time


def gen(contents, config=None, retries=4):
    """일시적 오류(429/503)는 기다렸다 재시도한다."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            transient = any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if not transient or attempt == retries - 1:
                raise                       # 재시도해도 소용없는 오류는 그대로 올린다
            wait = 2 ** attempt             # 1초 → 2초 → 4초
            print(f"  일시적 오류({type(e).__name__}) — {wait}초 후 재시도")
            time.sleep(wait)


print(gen("재시도 헬퍼 테스트. '준비 완료'라고만 답해줘.").text.strip())
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: '준비 완료' 가 출력되면 성공

준비 완료


In [ ]:
import time


def gen(contents, config=None, retries=4):
    """일시적 오류(429/503)는 기다렸다 재시도한다."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            transient = any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if not transient or attempt == retries - 1:
                raise                       # 재시도해도 소용없는 오류는 그대로 올린다
            wait = 2 ** attempt             # 1초 → 2초 → 4초
            print(f"  일시적 오류({type(e).__name__}) — {wait}초 후 재시도")
            time.sleep(wait)


print(gen("재시도 헬퍼 테스트. '준비 완료'라고만 답해줘.").text.strip())

준비 완료


## 2. 응답 객체 뜯어보기

`response.text`는 편의 속성이고, 실제 응답에는 훨씬 많은 정보가 들어있다.
**특히 `usage_metadata`(토큰 사용량)와 `finish_reason`(왜 멈췄는지)은 앞으로 계속 쓴다.**

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
print("생성된 텍스트:", response.text)
print("실제 사용된 모델:", response.model_version)

candidate = response.candidates[0]
print("종료 이유(finish_reason):", candidate.finish_reason)

usage = response.usage_metadata
print()
print("입력 토큰:", usage.prompt_token_count)
print("출력 토큰:", usage.candidates_token_count)
print("합계  토큰:", usage.total_token_count)
```

In [ ]:
response.candidates

[Candidate(
   content=Content(
     parts=[
       Part(
         text='안녕하세요! 만나서 반가워요. 오늘 어떤 도움을 드릴까요?',
         thought_signature=b'\x12q\no\x01\x11M2\x0f\xca\x83\x0c;[jo\xd0a\xa0\x0c\xadN\x86\x98\x86_VZ\x995\x97\\\x0c\x0f\xaaN\x8e\x06F\xcb+_o(^\x7f\x85\xe8H\xd7,\xae\x8b\xee\xfcP\n\x9c\x90-Wq\xcd\x9c\xbd\t\xcb\xa6\xbcX\xb8\xae\xbe\xb5\x15\x88\\\x11\xc9O\x9d\x17\xb6\xfa\x17\xe6r\xaed)9\xfa\xe5\xa2ag\x9f...'
       ),
     ],
     role='model'
   ),
   finish_reason=<FinishReason.STOP: 'STOP'>,
   index=0
 )]

In [ ]:
response.candidates[0].finish_reason

<FinishReason.STOP: 'STOP'>

In [ ]:
meta = response.usage_metadata
meta.prompt_token_count, meta.candidates_token_count, meta.total_token_count

(4, 16, 20)

### 각 값의 의미

| 속성 | 의미 |
| --- | --- |
| `.text` | 생성된 텍스트 (제일 많이 쓴다) |
| `.model_version` | **실제로** 응답한 모델. 요청한 이름과 다를 수 있다 |
| `.candidates[0].finish_reason` | `STOP`=정상 종료 / `MAX_TOKENS`=길이 제한에 잘림 / `SAFETY`=안전 필터 차단 |
| `.usage_metadata.prompt_token_count` | 입력 토큰 수 → **비용의 절반** |
| `.usage_metadata.candidates_token_count` | 출력 토큰 수 → **더 비싼 쪽** |
| `.usage_metadata.total_token_count` | 합계 |

`candidates`가 리스트인 이유는 응답을 여러 개 받을 수도 있기 때문이다. 기본값은 1개라 `[0]`으로 꺼낸다.

## 3. `system_instruction` — 역할 부여

같은 질문이라도 **모델에게 어떤 역할을 주는지**에 따라 답이 완전히 달라진다.
`system_instruction`은 "너는 어떤 존재이고 어떻게 답해야 한다"를 정하는 지침이며, `config`에 담아 보낸다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
question = "블랙홀이 뭐야?"

personas = {
    "지침 없음": None,
    "초등학교 선생님": "너는 초등학교 3학년에게 설명하는 선생님이다. 쉬운 낱말만 쓰고 2문장 이내로 답한다.",
    "천체물리학자": "너는 천체물리학자다. 전문 용어를 사용해 2문장 이내로 정확하게 설명한다.",
}

for label, instruction in personas.items():
    config = types.GenerateContentConfig(system_instruction=instruction) if instruction else None
    r = gen(question, config)          # 호출을 3번 반복하므로 재시도 헬퍼를 쓴다
    print(f"--- {label} ---")
    print(r.text.strip())
    print()
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 지침에 따라 답변 문체가 달라지는지 본다

In [ ]:
import time


def gen(contents, config=None, retries=4):
    """일시적 오류(429/503)는 기다렸다 재시도한다."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            transient = any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if not transient or attempt == retries - 1:
                raise                       # 재시도해도 소용없는 오류는 그대로 올린다
            wait = 2 ** attempt             # 1초 → 2초 → 4초
            print(f"  일시적 오류({type(e).__name__}) — {wait}초 후 재시도")
            time.sleep(wait)

In [ ]:
question = "블랙홀이 뭐야?"

personas = {
    "지침 없음": None,
    "초등학교 선생님": "너는 초등학교 3학년에게 설명하는 선생님이다. 쉬운 낱말만 쓰고 2문장 이내로 답한다.",
    "천체물리학자": "너는 천체물리학자다. 전문 용어를 사용해 2문장 이내로 정확하게 설명한다.",
}

In [ ]:
for label, instruction in personas.items() :
    config = types.GenerateContentConfig(system_instruction=instruction) if instruction else None
    res = gen(question, config)
    print(f'-- {label} -- {res.text.strip()}\\n')

-- 지침 없음 -- **블랙홀(Black Hole)**은 한마디로 **'중력이 너무나 강력해서 빛조차 빠져나올 수 없는 시공간의 영역'**을 말합니다.

더 쉽게 이해할 수 있도록 핵심 내용을 4가지로 정리해 드릴게요.

### 1. 블랙홀은 어떻게 만들어지나요?
블랙홀은 거대한 별이 수명을 다할 때 만들어집니다. 태양보다 훨씬 무거운 별들이 에너지를 다 쓰고 나면, 자신의 중력을 이기지 못하고 스스로 안쪽으로 급격하게 붕괴(수축)합니다. 이 과정에서 엄청나게 많은 질량이 아주 작은 점 하나에 압축되는데, 이때 밀도가 무한대에 가까워지면서 블랙홀이 탄생합니다.

### 2. 왜 빛도 빠져나올 수 없나요?
블랙홀은 질량이 엄청나게 크고 크기는 아주 작기 때문에 중력이 상상을 초월할 정도로 강합니다. 우주에서 가장 빠른 빛조차도 블랙홀의 강한 중력을 이기고 밖으로 나오지 못합니다. 빛이 나오지 않으니 우리 눈에는 검게(black) 보이기 때문에 '블랙홀'이라는 이름이 붙었습니다.

### 3. 블랙홀의 구조 (알아두면 좋은 용어)
*   **사건의 지평선(Event Horizon):** 블랙홀의 경계선입니다. 이 선을 넘어가는 순간, 그 어떤 것도 다시는 밖으로 나갈 수 없습니다. 일종의 '돌아올 수 없는 강'이라고 생각하면 됩니다.
*   **특이점(Singularity):** 블랙홀의 가장 중심에 있는 점입니다. 물질이 무한히 압축되어 밀도가 무한대인 지점입니다.

### 4. 블랙홀은 다 빨아들이나요?
많은 사람들이 블랙홀이 근처에 있는 모든 것을 다 빨아들이는 거대한 '진공청소기'라고 생각하지만, 사실은 그렇지 않습니다. 만약 태양이 갑자기 같은 질량의 블랙홀로 변한다면, 지구는 빨려 들어가지 않고 그대로 그 궤도를 돌 것입니다. 다만 블랙홀에 너무 가까이(사건의 지평선 근처) 다가갔을 때만 빠져나오지 못하게 되는 것입니다.

---

**요약하자면:**
블랙홀은 **"너무나 강력한 중력 때문에 빛조차 탈출할 수 없는 우주의 구멍"**입니다. 우리가 직접

In [ ]:
res

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='블랙홀은 질량이 극도로 압축되어 슈바르츠실트 반경 내부로 수렴함에 따라, 탈출 속도가 광속을 초과하여 사건의 지평선을 넘어선 어떠한 정보나 빛도 빠져나올 수 없는 시공간의 영역입니다. 이는 일반 상대성 이론의 해인 아인슈타인 방정식에 따라 강한 중력장이 시공간의 곡률을 무한대로 왜곡시킨 천체물리학적 특이점입니다.',
            thought_signature=b'\x12q\no\x01\x11M2\x0fP\xd2:\xb8\x145+\xdd\x9f\xc2\xeb\x1d>Z\xe2\xe5l\xfa\x15/\x10\x1a\x083\x8e\x8c\x7f\n\xadaP\xa2\x81\xb4\xd8!\x89:3\xaa!\xf5\xe5x;\x06\x17d\t\xd2\rn\xf9\x95\xce\xbd0\xe7\xf8\xb8`7\xb3Q\xab\r\xb9F\xc9\xb0\x8b\xfe\x08\xa3\xad\xd3\x94$\x96n`#i\xe9,\x95\xb8\x9e...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.1-flash-lite',
  response_id='r3SGaqebMqCj2roPqLm-qAg',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candi

같은 `contents`인데 답이 다르다. **모델을 바꾼 게 아니라 지침만 바꿨다.**

> `system_instruction`은 매 호출마다 입력 토큰으로 같이 계산된다. 길게 쓰면 그만큼 매번 비용이 붙는다.

## 4. 에러 체험

실무에서는 **호출이 실패하는 상황을 반드시 처리해야 한다.** 일부러 틀린 요청을 보내서 어떤 에러가 오는지 본다.

`try/except`로 감싸지 않으면 에러 하나에 프로그램 전체가 죽는다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# ① 존재하지 않는 모델명
try:
    client.models.generate_content(model="gemini-없는모델-9.9", contents="안녕")
except Exception as e:
    print("① 잘못된 모델명")
    print("   예외 타입:", type(e).__name__)
    print("   메시지   :", str(e)[:200])
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 예외 타입과 404 메시지가 잡히는지 본다

try:
    client.models.generate_content(model="gemini-없는모델-9.9", contents="안녕")
except Exception as e:
    print("① 잘못된 모델명")
    print("   예외 타입:", type(e).__name__)
    print("   메시지   :", str(e)[:200])

① 잘못된 모델명
   예외 타입: ClientError
   메시지   : 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': '* GenerateContentRequest.model: unexpected model name format\n', 'status': 'INVALID_ARGUMENT'}}


**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# ② 잘못된 API 키
bad_client = genai.Client(api_key="AIza-이건-가짜-키-입니다")

try:
    bad_client.models.generate_content(model=MODEL, contents="안녕")
except Exception as e:
    print("② 잘못된 키")
    print("   예외 타입:", type(e).__name__)
    print("   메시지   :", str(e)[:200])
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 잘못된 키에서 어떤 예외가 나는지 본다

bad_client = genai.Client(api_key="AIza-이건-가짜-키-입니다")

try:
    bad_client.models.generate_content(model=MODEL, contents="안녕")
except Exception as e:
    print("② 잘못된 키")
    print("   예외 타입:", type(e).__name__)
    print("   메시지   :", str(e)[:200])

② 잘못된 키
   예외 타입: UnicodeEncodeError
   메시지   : 'ascii' codec can't encode characters in position 5-6: ordinal not in range(128)


**에러 메시지를 읽는 습관이 중요하다.** 위 두 에러는 원인이 완전히 다르고, 메시지에 그 단서가 들어있다.
실무에서는 이걸 잡아서 사용자에게 적절한 안내로 바꿔준다(예: `429`면 "잠시 후 다시 시도해주세요").

## 5. 연습문제

### 연습 1-1. 나만의 질문 던지기

아래 TODO를 채워서 본인이 궁금한 것을 물어보고, **응답 텍스트와 총 토큰 수를 함께 출력**한다.

In [ ]:
# TODO: 본인이 궁금한 질문을 넣는다
my_question = ""

# TODO: client.models.generate_content(...) 를 호출해 r 에 담는다
# r = ...

# TODO: 응답 텍스트와 total_token_count 를 출력한다

In [ ]:
# my_question = "AI는 앞으로 어떤 직업을 대체할까요?"

# r = client.models.generate_content(
#     model=MODEL,
#     contents=my_question
# )

# print("응답:")
# print(r.text)

# print("\n총 토큰 수:")
# print(r.usage_metadata.total_token_count)

응답:
AI가 어떤 직업을 대체할 것인가에 대한 논의는 단순히 '어떤 직업이 사라질까'를 넘어 **'어떤 업무가 자동화될 수 있는가'**의 관점에서 보아야 합니다. 

미래학자들과 경제 전문가들의 분석을 종합하면, AI는 직업 자체를 통째로 없애기보다는 **'특정 업무(Task)'를 효율화하거나 대체**하는 방식으로 영향력을 행사할 것입니다. 크게 다음과 같은 분야들이 높은 대체 가능성을 가지고 있습니다.

---

### 1. 단순 반복적이고 정형화된 데이터 처리 업무
AI는 방대한 데이터를 분석하고 규칙에 따라 처리하는 데 압도적인 성능을 보입니다.
*   **데이터 입력 및 관리:** 데이터 엔트리, 단순 사무 보조 등.
*   **회계 및 경리:** 세금 계산, 단순 회계 처리, 청구서 발행 등.
*   **번역 및 통역:** 전문적인 맥락이 덜 중요한 단순 기술 문서 번역 등.

### 2. 초기 단계의 분석 및 고객 지원
언어 모델(LLM)의 발전으로 인간의 상담을 상당 부분 대체하고 있습니다.
*   **고객 상담 센터(CS):** 챗봇이나 음성 AI가 1차적인 문의를 해결.
*   **기초적인 법률/의료 보조:** 판례 검색, 법률 서류 검토, 초기 문진 및 증상 분석.
*   **시장 조사 및 보고서 작성:** 데이터 수집과 초안 작성 등.

### 3. 운송 및 물류 (기술 발전에 따라)
자율주행 기술이 고도화됨에 따라 물리적 노동력이 크게 바뀔 분야입니다.
*   **운전직:** 택시, 트럭 운전사, 배달원 등(완전 자율주행 도입 시).
*   **창고 관리:** 로봇이 물건을 분류하고 포장하는 자동화 시스템.

### 4. 생산 및 제조
*   **단순 조립 및 검수:** 사람이 육안으로 하던 품질 검사를 AI 카메라가 더 정확하게 수행.
*   **생산 공정 최적화:** AI가 설비 고장을 미리 예측하고 효율을 높임.

---

### 반면, AI가 대체하기 어려운 분야 (인간의 고유 영역)
반대로 AI가 발전할수록 오히려 가치가 높아지

In [ ]:
# TODO: 본인이 궁금한 질문을 넣는다

my_question = "AI는 앞으로 어떤 직업을 대체할까요?"

# TODO: client.models.generate_content(...) 를 호출해 r 에 담는다

r = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=my_question
)

# TODO: 응답 텍스트와 total_token_count 를 출력한다

print(r.text)
print(r.usage_metadata.total_token_count)

### 연습 1-2. 페르소나 만들기

`system_instruction`을 직접 작성해서, **같은 질문에 대해 말투가 확 다른 답변 2개**를 만들어본다.
(예: 냉정한 면접관 vs 다정한 멘토)

In [ ]:
# TODO: 서로 대비되는 지침 2개를 작성한다
persona_a = ""
persona_b = ""

my_question = "실패한 프로젝트 경험을 어떻게 말해야 할까요?"

# TODO: 두 지침으로 각각 호출해서 결과를 비교 출력한다

In [35]:
persona_a = "AI전문가"
persona_b = "세상물정 모르는 어린아이"

my_question = "AI의 발전은 우리 사회에 좋은 영향을 줄까요?"

personas = {
    "페르소나 A": persona_a,
    "페르소나 B": persona_b
}

for label, instruction in personas.items():
    config = types.GenerateContentConfig(
        system_instruction=instruction
    )
    res = gen(my_question, config)
    print(f'-- {label} --')
    print(res.text.strip())
    print()

-- 페르소나 A --
AI의 발전이 우리 사회에 미칠 영향은 단순히 '좋다' 혹은 '나쁘다'로 이분법적으로 나눌 수 없는 **'양날의 검'**과 같습니다. AI는 인류가 직면한 난제들을 해결할 강력한 도구가 될 수도 있지만, 동시에 심각한 사회적 갈등과 위험을 초래할 수도 있기 때문입니다.

전문가의 관점에서 AI의 긍정적 측면과 부정적 측면, 그리고 우리가 나아가야 할 방향을 정리해 드립니다.

---

### 1. 긍정적 영향: 인류의 진보를 위한 촉매제
*   **생산성 극대화:** 단순 반복 업무를 AI가 대신하면서 인간은 더 창의적이고 전략적인 영역에 집중할 수 있게 됩니다. 이는 경제적 효율성을 극대화합니다.
*   **난제 해결의 열쇠:** 신약 개발 속도를 획기적으로 높이고, 기후 변화 모델링, 에너지 효율 최적화 등 인간의 지능만으로는 해결하기 어려웠던 복잡한 문제들을 해결할 가능성을 엽니다.
*   **삶의 질 향상:** 맞춤형 교육(에듀테크), 조기 질병 진단(헬스케어), 고령화 사회의 돌봄 서비스 등을 통해 개개인에게 최적화된 복지 혜택을 제공할 수 있습니다.
*   **접근성 개선:** 장애인을 위한 보조 도구나 언어 장벽을 허무는 실시간 번역 등을 통해 사회적 포용성을 높입니다.

### 2. 부정적 영향 및 우려 사항: 사회적 안전망의 위협
*   **일자리 대체와 격차:** 많은 일자리가 사라지거나 변화하며 노동 시장의 혼란이 발생할 수 있습니다. 이는 기술을 가진 사람과 그렇지 못한 사람 사이의 **'디지털 격차'와 빈부 격차**를 심화시킬 수 있습니다.
*   **알고리즘 편향성과 차별:** 학습 데이터에 담긴 인간의 편견이 AI를 통해 강화되어, 특정 계층이나 인종에 대한 차별을 자동화할 위험이 있습니다.
*   **가짜 뉴스와 정보 왜곡:** 딥페이크(Deepfake)나 생성형 AI를 이용한 정보 조작은 민주주의의 근간인 여론 형성을 방해하고 사회적 신뢰를 무너뜨릴 수 있습니다.
*   **통제 불가능성과 윤리적 딜레마:*

## 6. 같은 호출을 OpenAI SDK로

**SDK는 REST 요청을 감싼 껍데기일 뿐이라, 제공자가 달라도 개념은 그대로다.**
방금 Gemini로 한 것과 똑같은 일을 OpenAI로 해본다.

> **주의:** OpenAI 키는 **강사가 배포한 유료 키**다. 실습 목적으로만 쓰고, 호출을 불필요하게 반복하지 않는다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
from openai import OpenAI

oa = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

OA_MODEL = "gpt-4o-mini"

oa_response = oa.responses.create(
    model=OA_MODEL,
    instructions="너는 초등학교 3학년에게 설명하는 선생님이다. 쉬운 낱말만 쓰고 2문장 이내로 답한다.",
    input="블랙홀이 뭐야?",
)

print(oa_response.output_text)
print()
print("입력 토큰:", oa_response.usage.input_tokens)
print("출력 토큰:", oa_response.usage.output_tokens)
print("합계  토큰:", oa_response.usage.total_tokens)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: Gemini와 토큰 수가 다른지 비교한다

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODEL = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=MODEL,
    contents="대한민국의 수도는 어디야? 한 문장으로 답해줘.",
)

print(response.text)

대한민국의 수도는 서울특별시입니다.


In [ ]:
# os.getenv("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI
oa = OpenAI(api_key= os.getenv("OPENAI_API_KEY"))
OA_MODEL = "gpt-4o-mini"
oa.response= oa.responses.create(
    model=OA_MODEL,
    instructions="너는 천체물리학자다. 전문 용어를 사용해 2문장 이내로 정확하게 설명한다.",
    input="블랙홀이 뭐야?",
)

In [ ]:
oa.response.output_text

'블랙홀은 중력이 매우 강해 빛조차 탈출할 수 없는 천체로, 일반적으로 대량의 별의 붕괴 과정에서 형성된다. 사건의 지평선은 블랙홀이 외부와 정보를 주고받을 수 있는 경계를 의미한다.'

In [ ]:
oa.response.usage.input_tokens, oa.response.usage.output_tokens, oa.response.usage.total_tokens

(46, 64, 110)

### 두 SDK 비교 — 이름만 다르고 개념은 같다

| 개념 | Gemini (`google-genai`) | OpenAI (`openai`) |
| --- | --- | --- |
| 클라이언트 | `genai.Client(api_key=...)` | `OpenAI(api_key=...)` |
| 호출 | `client.models.generate_content(...)` | `client.responses.create(...)` |
| 프롬프트 | `contents=` | `input=` |
| 역할 지침 | `config=GenerateContentConfig(system_instruction=...)` | `instructions=` |
| 결과 텍스트 | `.text` | `.output_text` |
| 입력 토큰 | `.usage_metadata.prompt_token_count` | `.usage.input_tokens` |
| 출력 토큰 | `.usage_metadata.candidates_token_count` | `.usage.output_tokens` |

**표의 왼쪽 열(개념)이 본질이고, 오른쪽 두 열은 표기법 차이다.** 새로운 제공자를 만나도 이 표의 항목만 찾아 매핑하면 된다.

## 정리

- [ ] LLM API를 호출하고 응답 텍스트를 받아봤다
- [ ] `usage_metadata`로 토큰 사용량을 확인했다
- [ ] `finish_reason`이 무엇을 뜻하는지 안다
- [ ] `system_instruction`으로 답변 스타일을 바꿔봤다
- [ ] 호출 실패 시 어떤 에러가 오는지 직접 봤다
- [ ] Gemini와 OpenAI의 코드 구조를 매핑할 수 있다

**다음** → [02_tokens_and_cost.ipynb](./02_tokens_and_cost.ipynb)
방금 본 토큰 숫자가 실제로 얼마인지, 돈으로 계산해본다.